In [ ]:
import os
os.environ['TORCHDYNAMO_DISABLE'] = '1'
# os.environ['HF_HUB_OFFLINE'] = '1'
os.chdir('/home/maki')
import re
import ast
import json
import time
import math
import random
import operator
import itertools
import numpy as np
import pandas as pd
from pathlib import Path
from itertools import permutations, product
from collections import Counter

In [ ]:
from huggingface_hub import login

# HF_TOKEN = userdata.get("HF_TOKEN")
HF_TOKEN = "HT"
# print("HF_TOKEN задан")
login(token=HF_TOKEN, add_to_git_credential=True)
print(" HuggingFace авторизован")

DRIVE_DIR = Path("./gemma-countdown-sft")
DRIVE_DIR.mkdir(parents=True, exist_ok=True)
print(f" Рабочая папка: {DRIVE_DIR}")

In [ ]:
CFG = {
    # Модель
    "student_model" : "./models",
    "hf_hub_model"  : "mi44dll/gemma-3-1b-countdown-sft",

    # Данные
    "dataset_name"   : "HuggingFaceTB/Countdown-Task-GOLD",
    "dataset_config": "verified_Qwen3-4B-Instruct-2507",
    "jsonl_path" : "./data/countdown_qwen3_4b.jsonl"
    # "dataset_config" : "all",
    "val_split"      : 0.05,
    "synth_5_6_count": 0,

    # Tokenizer / sequence
    "max_seq_length" : 1024,

    # Обучение
    "learning_rate"  : 2e-4,
    "batch_size"     : 4,   
    "grad_accum"     : 4,   
    "num_epochs"     : 1,
    "warmup_ratio"   : 0.05,
    "lr_scheduler"   : "cosine",
    "optim"          : "adamw_8bit",
    "max_grad_norm"  : 1.0,
    "fp16"           : False, 
    "bf16"           : True,
    "weight_decay"   : 0.01,

    # Валидация и сохранение
    "eval_steps"     : 200,
    "save_steps"     : 70,
    "save_total_limit": 3,
    "logging_steps"  : 10,
    "load_best"      : False,

    # Inference
    "max_new_tokens" : 1024,
    "batch_inf"      : 16,
    "test_csv"       : "test_public.csv",
    "submission_csv" : str(DRIVE_DIR / "submission.csv"),

    # Пути
    "output_dir"     : str(DRIVE_DIR / "checkpoints"),
    "seed"           : 42,
}

random.seed(CFG["seed"])
np.random.seed(CFG["seed"])
print(" Конфигурация задана")
print(json.dumps(CFG, indent=2))

In [ ]:
def solve_countdown(nums, target):
    """
    Брутфорс решатель Countdown.
    Возвращает строку-выражение или None если решения нет.
    Работает за <1 сек для 6 чисел.
    """
    ops = [operator.add, operator.sub, operator.mul, operator.truediv]
    op_syms = {operator.add:'+', operator.sub:'-',
               operator.mul:'*', operator.truediv:'/'}

    def solve(numbers):
        if len(numbers) == 1:
            val, expr = numbers[0]
            if abs(val - target) < 1e-9 and val == int(val):
                return expr
            return None

        for i in range(len(numbers)):
            for j in range(len(numbers)):
                if i == j:
                    continue
                a_val, a_expr = numbers[i]
                b_val, b_expr = numbers[j]
                rest = [numbers[k] for k in range(len(numbers))
                        if k != i and k != j]

                for op in ops:
                    # Пропускаем деление на ноль
                    if op == operator.truediv and abs(b_val) < 1e-9:
                        continue
                    # Пропускаем нецелые результаты деления
                    if op == operator.truediv:
                        if abs(a_val % b_val) > 1e-9:
                            continue

                    result = op(a_val, b_val)
                    sym = op_syms[op]

                    # Формируем выражение с минимальными скобками
                    if op in (operator.mul, operator.truediv):
                        a_str = (f"({a_expr})"
                                 if any(s in a_expr for s in ('+','-'))
                                 else a_expr)
                        b_str = (f"({b_expr})"
                                 if any(s in b_expr for s in ('+','-'))
                                 else b_expr)
                    else:
                        a_str = a_expr
                        b_str = (f"({b_expr})"
                                 if op == operator.sub and
                                 any(s in b_expr for s in ('+','-'))
                                 else b_expr)
                    new_expr = f"{a_str} {sym} {b_str}"
                    new_nums = rest + [(result, new_expr)]

                    found = solve(new_nums)
                    if found is not None:
                        return found
        return None

    numbered = [(float(n), str(n)) for n in nums]
    return solve(numbered)

In [ ]:
from datasets import load_dataset, Dataset, concatenate_datasets

def validate_equation(eq_str, nums, target):
    """
    Проверяет уравнение по трём правилам submission:
      1. Только числа из nums (каждое не более одного раза)
      2. Только операции +, -, *, /
      3. eval(eq) == target
    """
    try:
        # Правило 2: только допустимые символы
        allowed = set("0123456789 +-*/().")
        if not set(eq_str) <= allowed:
            return False

        # Правило 1: числа совпадают
        nums_in_eq = [int(x) for x in re.findall(r'\d+', eq_str)]
        if sorted(nums_in_eq) != sorted(nums):
            # Допускаем использование подмножества чисел
            nums_avail = sorted(nums)
            for n in nums_in_eq:
                if n not in nums_avail:
                    return False
                nums_avail.remove(n)

        # Правило 3: математически верно
        result = eval(eq_str)
        return abs(result - target) < 1e-6

    except Exception:
        return False


def extract_equation(assistant_content):
    """
    Извлекает уравнение из <answer>...</answer>.
    Формат: 'expr = target' → возвращает только 'expr'.
    """
    m = re.search(r'<answer>(.*?)</answer>',
                  assistant_content, re.DOTALL)
    if not m:
        return None
    raw = m.group(1).strip()
    # Убираем '= target' если есть
    if '=' in raw:
        return raw.split('=')[0].strip()
    return raw


def load_and_prepare_dataset(cfg):
    print(f"\nЗагружаем {cfg['dataset_name']} [{cfg['dataset_config']}]...")
    ds = load_dataset(cfg["dataset_name"], cfg["dataset_config"],
                      split="train")
    print(f"  Загружено: {len(ds)} примеров")
    print(f"  Колонки: {ds.column_names}")

    print("\nФильтруем некорректные ответы...")

    def is_valid(example):
        msgs = example["messages"]
        asst = next((m for m in msgs if m["role"] == "assistant"), None)
        if asst is None:
            return False
        eq = extract_equation(asst["content"])
        if eq is None:
            return False
        return validate_equation(eq, example["nums"], example["target"])

    ds_filtered = ds.filter(is_valid, num_proc=1)
    n_dropped = len(ds) - len(ds_filtered)
    print(f"  Осталось: {len(ds_filtered)} "
          f"(отфильтровано: {n_dropped})")

    print("\nДедупликация...")
    seen = set()
    keep_idx = []
    for i, ex in enumerate(ds_filtered):
        key = (ex["target"], tuple(sorted(ex["nums"])))
        if key not in seen:
            seen.add(key)
            keep_idx.append(i)
    ds_deduped = ds_filtered.select(keep_idx)
    print(f"  После дедупликации: {len(ds_deduped)} "
          f"(удалено дублей: {len(ds_filtered) - len(ds_deduped)})")

    # ── Статистика ───────────────────────────────────────────────────
    nums_dist = Counter(len(ex["nums"]) for ex in ds_deduped)
    print(f"\n  Распределение по кол-ву чисел:")
    for k in sorted(nums_dist):
        print(f"    {k} чисел: {nums_dist[k]:>5} "
              f"({nums_dist[k]/len(ds_deduped)*100:.1f}%)")

    # ── Синтетика 5-6 чисел ──────────────────────────────────────────
    # ── Загрузка синтетических данных из JSONL файла ─────────────────
    print(f"\nЗагружаем синтетические примеры из {cfg["json_path"]...")
    synth_examples = []
    
    jsonl_path = cfg["json_path"]
    try:
        with open(jsonl_path, 'r', encoding='utf-8') as f:
            for line_num, line in enumerate(f, 1):
                line = line.strip()
                if line:
                    try:
                        example = json.loads(line)
                        # Проверяем, что пример имеет нужную структуру
                        if all(k in example for k in ["target", "nums", "messages"]):
                            synth_examples.append(example)
                        else:
                            print(f"    Предупреждение: строка {line_num} пропущена (не хватает полей)")
                    except json.JSONDecodeError as e:
                        print(f"    Ошибка в строке {line_num}: {e}")
        
        print(f"  Загружено: {len(synth_examples)} синтетических примеров")
        
        # Дополнительная фильтрация синтетических данных
        print(f"  Фильтруем синтетические примеры...")
        valid_synth = []
        for ex in synth_examples:
            asst = next((m for m in ex["messages"] if m["role"] == "assistant"), None)
            if asst is None:
                continue
            eq = extract_equation(asst["content"])
            if eq is None:
                continue
            if validate_equation(eq, ex["nums"], ex["target"]):
                valid_synth.append(ex)
        
        filtered_out = len(synth_examples) - len(valid_synth)
        if filtered_out > 0:
            print(f"    Отфильтровано некорректных: {filtered_out}")
        
        ds_synth = Dataset.from_list(valid_synth)
        print(f"  Итого синтетических примеров: {len(ds_synth)}")
        
    except FileNotFoundError:
        print(f"  ОШИБКА: Файл {jsonl_path} не найден!")
        print(f"  Создаём пустой датасет вместо синтетических примеров")
        ds_synth = Dataset.from_list([])
    except Exception as e:
        print(f"  ОШИБКА при загрузке {jsonl_path}: {e}")
        print(f"  Создаём пустой датасет вместо синтетических примеров")
        ds_synth = Dataset.from_list([])

    # ── Объединяем ───────────────────────────────────────────────────
    ds_combined = concatenate_datasets([ds_deduped, ds_synth])
    ds_combined = ds_combined.shuffle(seed=cfg["seed"])
    print(f"\n  Итоговый датасет: {len(ds_combined)} примеров")

    # ── Train / Val split ────────────────────────────────────────────
    val_size = int(len(ds_combined) * cfg["val_split"])
    split = ds_combined.train_test_split(
        test_size=val_size, seed=cfg["seed"]
    )
    ds_train = split["train"]
    ds_val   = split["test"]

    print(f"\n  Train: {len(ds_train)} примеров")
    print(f"  Val:   {len(ds_val)} примеров")

    print("\n  Итоговое распределение (train):")
    nums_dist_final = Counter(len(ex["nums"]) for ex in ds_train)
    for k in sorted(nums_dist_final):
        print(f"    {k} чисел: {nums_dist_final[k]:>5} "
              f"({nums_dist_final[k]/len(ds_train)*100:.1f}%)")

    return ds_train, ds_val

In [ ]:
import torch
from unsloth.chat_templates import train_on_responses_only
from trl import SFTTrainer, SFTConfig
from transformers import EarlyStoppingCallback

In [ ]:
from unsloth import FastModel
def load_model(cfg):
    """
    Загружает Gemma-3-1b-it через Unsloth с 4-bit квантованием.
    Применяет LoRA адаптеры.
    """
    model, tokenizer = FastModel.from_pretrained(
        model_name      = cfg["student_model"],
        max_seq_length  = cfg["max_seq_length"],
        load_in_4bit    = True,
        token           = HF_TOKEN,
        local_files_only = True,
    )

    model = FastModel.get_peft_model(
        model,
        finetune_vision_layers    = False,
        finetune_language_layers  = True,
        finetune_attention_modules = True,
        finetune_mlp_modules      = True,
        r           = 16,
        lora_alpha  = 16,
        lora_dropout = 0,
        bias        = "none",
        use_gradient_checkpointing = "unsloth",
        random_state = cfg["seed"],
    )
    
    print(f" Модель загружена: {cfg['student_model']}")
    print(f"  Параметров: {sum(p.numel() for p in model.parameters()):,}")

    trainable = sum(p.numel() for p in model.parameters()
                    if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    print(f"  LoRA trainable: {trainable:,} "
          f"({trainable/total*100:.2f}% от всех параметров)")
    print(f"  r=8, alpha=8, dropout=0")

    return model, tokenizer

# Train 

In [ ]:
cfg = CFG

In [ ]:
ds_train, ds_val = load_and_prepare_dataset(cfg)

In [ ]:
model, tokenizer = load_model(cfg)

In [ ]:
training_args = SFTConfig(
    output_dir             = cfg["output_dir"],

    num_train_epochs       = cfg["num_epochs"],

    per_device_train_batch_size  = cfg["batch_size"],
    per_device_eval_batch_size   = cfg["batch_size"],
    gradient_accumulation_steps  = cfg["grad_accum"],

    learning_rate          = cfg["learning_rate"],
    optim                  = cfg["optim"],
    weight_decay           = cfg["weight_decay"],
    max_grad_norm          = cfg["max_grad_norm"],
    warmup_ratio           = cfg["warmup_ratio"],
    lr_scheduler_type      = cfg["lr_scheduler"],

    fp16                   = cfg['fp16'],
    bf16                   = cfg['bf16'],

    max_seq_length         = cfg["max_seq_length"],

    packing                = True,

    eval_strategy          = "no",
    # eval_steps             = cfg["eval_steps"],

    save_strategy          = "steps",
    save_steps             = cfg["save_steps"],
    save_total_limit       = cfg["save_total_limit"],
    load_best_model_at_end = cfg["load_best"],
    metric_for_best_model  = "eval_loss",
    greater_is_better      = False,

    logging_steps          = cfg["logging_steps"],
    report_to              = "none",

    dataloader_num_workers = 0,
    dataloader_pin_memory  = False,
    dataset_num_proc       = 0,
    seed                   = cfg["seed"],

    push_to_hub            = False,
    hub_model_id           = cfg["hf_hub_model"],
)

In [ ]:
def formatting_func(examples):
    """
    Работает и с одним примером (dict), и с батчем (dict of lists).
    Unsloth вызывает её по-разному в зависимости от контекста.
    """
    if isinstance(examples["messages"][0], dict):
        all_messages = [examples["messages"]]
    else:
        all_messages = examples["messages"]

    texts = []
    for msgs in all_messages:
        text = tokenizer.apply_chat_template(
            msgs,
            tokenize=False,
            add_generation_prompt=False,
        )
        texts.append(text)
    return texts

In [ ]:
from transformers import TrainerCallback

class PrintLossCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is None:
            return
        step = state.global_step
        loss = logs.get("loss", "")
        eval_loss = logs.get("eval_loss", "")
        lr = logs.get("learning_rate", "")
        
        if eval_loss:
            print(f"  step={step:>4} | train_loss={loss:.4f} | "
                  f"eval_loss={eval_loss:.4f} | lr={lr:.2e}")
        elif loss:
            print(f"  step={step:>4} | train_loss={loss:.4f} | lr={lr:.2e}")

In [ ]:
trainer = SFTTrainer(
    model            = model,
    processing_class = tokenizer,
    train_dataset    = ds_train,
    eval_dataset     = ds_val,
    args             = training_args,
    formatting_func  = formatting_func,
    callbacks        = [
        PrintLossCallback(),
    ],
)

# Маскируем промпт — учим только ответы assistant
# Для Gemma: response_part = "<start_of_turn>model\n"
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<start_of_turn>user\n",
    response_part    = "<start_of_turn>model\n",
)

In [ ]:
print(f"\nЭффективный batch size: "
      f"{cfg['batch_size'] * cfg['grad_accum']}")
print(f"Шагов на эпоху: "
      f"~{len(ds_train) // (cfg['batch_size'] * cfg['grad_accum'])}")
print(f"Всего шагов: ~"
      f"{len(ds_train) // (cfg['batch_size'] * cfg['grad_accum']) * cfg['num_epochs']}")
print(f"\nНачинаем обучение...\n")

start_time = time.time()
result = trainer.train()
elapsed = time.time() - start_time

In [ ]:
print(f"\n Обучение завершено за {elapsed/60:.1f} минут")
print(f"  Train loss: {result.training_loss:.4f}")
print(f"  Шагов:      {result.global_step}")

In [ ]:
def run_inference_llamacpp(cfg):
    import requests
    import concurrent.futures

    print("\n" + "="*60)
    print("INFERENCE НА ТЕСТОВЫХ ДАННЫХ (llama-server)")
    print("="*60)

    test_df = pd.read_csv(cfg["test_csv"])
    test_df["nums"] = test_df["nums"].apply(ast.literal_eval)
    print(f"  Тестовых примеров: {len(test_df)}")

    SYSTEM_PROMPT = (
        "You are a helpful assistant. You first think about the reasoning "
        "process in the mind and then provide the user with the answer."
    )

    def query_server(row):
        nums   = row["nums"]
        target = int(row["target"])
        prompt = (
            f"<start_of_turn>user\n"
            f"{SYSTEM_PROMPT}\n\n"
            f"Using the numbers {nums}, create an equation that equals {target}. "
            f"You can use basic arithmetic operations (+, -, *, /) "
            f"and each number can only be used once. Show your work in "
            f"<think> </think> tags. And return the final equation and answer "
            f"in <answer> </answer> tags, for example "
            f"<answer> (1 + 2) / 3 </answer>."
            f"<end_of_turn>\n<start_of_turn>model\n"
        )
        try:
            response = requests.post(
                "http://localhost:8080/completion",
                json={
                    "prompt"      : prompt,
                    "n_predict"   : 768,
                    "temperature" : 0,
                    "stop"        : ["<end_of_turn>", "<eos>"],
                },
                timeout=120,
            )
            return response.json()["content"]
        except Exception as e:
            print(f"  Ошибка: {e}")
            return ""

    equations  = []
    n_correct  = 0
    n_no_answer = 0
    start = time.time()
    BATCH = 16

    print(f"\nЗапускаем inference (batch={BATCH})...")

    for batch_start in range(0, len(test_df), BATCH):
        batch = test_df.iloc[batch_start : batch_start + BATCH]
        rows  = [row for _, row in batch.iterrows()]

        with concurrent.futures.ThreadPoolExecutor(max_workers=BATCH) as executor:
            outputs = list(executor.map(query_server, rows))

        for generated, row in zip(outputs, rows):
            eq = extract_equation(generated)
            if eq is None:
                n_no_answer += 1
                eq = solve_countdown(row["nums"], int(row["target"])) or ""
            else:
                if validate_equation(eq, row["nums"], int(row["target"])):
                    n_correct += 1
            equations.append(eq)

        done    = min(batch_start + BATCH, len(test_df))
        elapsed = time.time() - start
        speed   = done / elapsed if elapsed > 0 else 0
        eta     = (len(test_df) - done) / speed if speed > 0 else 0
        print(f"  [{done:>4}/{len(test_df)}] "
              f"speed={speed:.2f} ex/s  "
              f"ETA={eta/60:.1f}m  "
              f"no_answer={n_no_answer}")

    elapsed = time.time() - start
    print(f"\n Inference завершён за {elapsed/60:.1f} минут")
    print(f"  Без ответа: {n_no_answer}")
    print(f"  Верных:     {n_correct}/{len(test_df)}")
    print(f"  Accuracy:   {n_correct/len(test_df)*100:.1f}%")

    submission = pd.DataFrame({
        "id":       test_df["id"].values,
        "equation": equations
    })
    submission.to_csv(cfg["submission_csv"], index=False)
    print(f"\n Submission сохранён: {cfg['submission_csv']}")
    print(submission.head(10).to_string(index=False))

    return submission

In [ ]:
submission = run_inference_llamacpp(CFG)